In [1]:
import pandas as pd
import chemsource
import os
import sys

sys.path.append(os.path.abspath("../src"))
from harmonization import (
    harmonize_automated_classification,
    harmonize_manual_classification,
)


In [2]:
classified_drug_library_data_path = "../data/drug_library/validation_data_classified_all_3_methods.csv"

harmonized_automated = harmonize_automated_classification(classified_drug_library_data_path)
harmonized_manual = harmonize_manual_classification(classified_drug_library_data_path)

drug_library_text = pd.read_csv(classified_drug_library_data_path)
drug_library_text["FEATURE_ID"] = drug_library_text.index
drug_library_text = drug_library_text[["FEATURE_ID", 
    "name_used", 
    "text"]].rename(columns={
    "name_used": "NAME",
    "text": "TEXT"})

openai_api_key = open("../secrets/openai_api_key.txt").read().strip()
ncbi_api_key = open("../secrets/ncbi_api_key.txt").read().strip()

chem = chemsource.ChemSource(
    model_api_key=openai_api_key,
    ncbi_key=ncbi_api_key,
    model="gpt-4o",
    clean_output=True,
    allowed_categories=["MEDICAL", "FOOD", "INDUSTRIAL", "PERSONAL CARE", "ENDOGENOUS", "INFO"]
)

chem.top_p = 0.0000001



In [4]:
prompt = "You are a helpful scientist that will classify the provided compound \
COMPOUND_NAME using only the information provided as any combination of the \
following: MEDICAL, ENDOGENOUS, FOOD, PERSONAL CARE, INDUSTRIAL. Note that \
MEDICAL refers to compounds actively used as approved medications in \
humans or in late-stage clinical trials in humans. Note that ENDOGENOUS \
refers to compounds that are produced by the human body specifically. \
ENDOGENOUS excludes essential nutrients that cannot be synthesized by the \
human body. Note that FOOD refers to compounds present in natural food items \
or food additives. Note that PERSONAL CARE refers to non-medicated compounds \
typically used for activities such as skincare, beauty, and fitness. Note \
that INDUSTRIAL should be used only for synthetic compounds not used as a \
contributing ingredient in the medical, personal care, or food industries. \
Specify INFO instead if more information is needed. DO NOT MAKE ANY \
ASSUMPTIONS, USE ONLY THE INFORMATION PROVIDED AFTER THE COMPOUND NAME \
BY THE USER. A classification of INFO will also be rewarded when \
correctly applied and is strongly encouraged if information is of poor \
quality, if there is not enough information, or if you are not completely \
confident in your answer.  Provide the output as a plain text separated \
by commas, and provide only the categories listed (either list a \
combination of INDUSTRIAL, ENDOGENOUS, PERSONAL CARE, MEDICAL, FOOD or \
list INFO), with no justification. Provided Information:\n"

prompt_industrial_strict = "You are a helpful scientist that will \
classify the provided compound COMPOUND_NAME using only the \
information provided as any combination of the \
following: MEDICAL, ENDOGENOUS, FOOD, PERSONAL CARE, INDUSTRIAL. Note that \
MEDICAL refers to compounds actively used as approved medications in \
humans or in late-stage clinical trials in humans. Note that ENDOGENOUS \
refers to compounds that are produced by the human body specifically. \
ENDOGENOUS excludes essential nutrients that cannot be synthesized by the \
human body. Note that FOOD refers to compounds present in natural food items \
or food additives. Note that PERSONAL CARE refers to non-medicated compounds \
typically used for activities such as skincare, beauty, and fitness. Note \
that INDUSTRIAL should be used only for synthetic compounds not used as a \
contributing ingredient in the medical, personal care, or food industries. \
Do NOT classify veterinary medications or any types of \
pesticides under the INDUSTRIAL \
category. Specify INFO instead if more information is needed. DO NOT MAKE ANY \
ASSUMPTIONS, USE ONLY THE INFORMATION PROVIDED AFTER THE COMPOUND NAME \
BY THE USER. A classification of INFO will also be rewarded when \
correctly applied and is strongly encouraged if information is of poor \
quality, if there is not enough information, or if you are not completely \
confident in your answer.  Provide the output as a plain text separated \
by commas, and provide only the categories listed (either list a \
combination of INDUSTRIAL, ENDOGENOUS, PERSONAL CARE, MEDICAL, FOOD or \
list INFO), with no justification. Provided Information:\n"

prompt_industrial_loose = "You are a helpful scientist that will \
classify the provided compound COMPOUND_NAME using only the \
information provided as any combination of the \
following: MEDICAL, ENDOGENOUS, FOOD, PERSONAL CARE, INDUSTRIAL. Note that \
MEDICAL refers to compounds actively used as approved medications in \
humans or in late-stage clinical trials in humans. Note that ENDOGENOUS \
refers to compounds that are produced by the human body specifically. \
ENDOGENOUS excludes essential nutrients that cannot be synthesized by the \
human body. Note that FOOD refers to compounds present in natural food items \
or food additives. Note that PERSONAL CARE refers to non-medicated compounds \
typically used for activities such as skincare, beauty, and fitness. \
Note that INDUSTRIAL should be used for all compounds that \
are used in any type of industrial process. Specify INFO instead if \
more information is needed. DO NOT MAKE ANY \
ASSUMPTIONS, USE ONLY THE INFORMATION PROVIDED AFTER THE COMPOUND NAME \
BY THE USER. A classification of INFO will also be rewarded when \
correctly applied and is strongly encouraged if information is of poor \
quality, if there is not enough information, or if you are not completely \
confident in your answer.  Provide the output as a plain text separated \
by commas, and provide only the categories listed (either list a \
combination of INDUSTRIAL, ENDOGENOUS, PERSONAL CARE, MEDICAL, FOOD or \
list INFO), with no justification. Provided Information:\n"


prompt_veterinary_looped_in = "You are a helpful scientist \
that will classify the provided compound COMPOUND_NAME \
using only the information provided as any combination of the \
following: MEDICAL, ENDOGENOUS, FOOD, PERSONAL CARE, INDUSTRIAL. Note that \
MEDICAL refers to compounds actively used as approved medications in \
humans or in late-stage clinical trials in humans and also includes \
any veterinary drugs used to treat pets, livestock, or other animals. \
Note that ENDOGENOUS refers to compounds \
that are produced by the human body specifically. \
ENDOGENOUS excludes essential nutrients that cannot be synthesized by the \
human body. Note that FOOD refers to compounds present in natural food items \
or food additives. Note that PERSONAL CARE refers to non-medicated compounds \
typically used for activities such as skincare, beauty, and fitness. Note \
that INDUSTRIAL should be used only for synthetic compounds not used as a \
contributing ingredient in the medical, personal care, or food industries. \
Specify INFO instead if more information is needed. DO NOT MAKE ANY \
ASSUMPTIONS, USE ONLY THE INFORMATION PROVIDED AFTER THE COMPOUND NAME \
BY THE USER. A classification of INFO will also be rewarded when \
correctly applied and is strongly encouraged if information is of poor \
quality, if there is not enough information, or if you are not completely \
confident in your answer.  Provide the output as a plain text separated \
by commas, and provide only the categories listed (either list a \
combination of INDUSTRIAL, ENDOGENOUS, PERSONAL CARE, MEDICAL, FOOD or \
list INFO), with no justification. Provided Information:\n"


prompt_industrial_strict_medical_loose = "You are a helpful scientist \
that will classify the provided compound COMPOUND_NAME using only the \
information provided as any combination of the \
following: MEDICAL, ENDOGENOUS, FOOD, PERSONAL CARE, INDUSTRIAL. Note that \
MEDICAL refers to any kind of medications used in humans or animals and also \
includes any veterinary drugs used to \
treat pets or livestock. Note that ENDOGENOUS \
refers to compounds that are produced by the human body specifically. \
ENDOGENOUS excludes essential nutrients that cannot be synthesized by the \
human body. Note that FOOD refers to compounds present in natural food items \
or food additives. Note that PERSONAL CARE refers to non-medicated compounds \
typically used for activities such as skincare, beauty, and fitness. Note \
that INDUSTRIAL should be used only for synthetic compounds not used as a \
contributing ingredient in the medical, personal care, or food industries. \
Do NOT classify veterinary medications or any types of \
pesticides under the INDUSTRIAL \
category. Specify INFO instead if more information is needed. DO NOT MAKE ANY \
ASSUMPTIONS, USE ONLY THE INFORMATION PROVIDED AFTER THE COMPOUND NAME \
BY THE USER. A classification of INFO will also be rewarded when \
correctly applied and is strongly encouraged if information is of poor \
quality, if there is not enough information, or if you are not completely \
confident in your answer.  Provide the output as a plain text separated \
by commas, and provide only the categories listed (either list a \
combination of INDUSTRIAL, ENDOGENOUS, PERSONAL CARE, MEDICAL, FOOD or \
list INFO), with no justification. Provided Information:\n"





prompt_no_description = "You are a helpful scientist that will classify the provided compound \
COMPOUND_NAME using only the information provided as any combination of the \
following: MEDICAL, ENDOGENOUS, FOOD, PERSONAL CARE, INDUSTRIAL. \
Specify INFO instead if more information is needed. DO NOT MAKE ANY \
ASSUMPTIONS, USE ONLY THE INFORMATION PROVIDED AFTER THE COMPOUND NAME \
BY THE USER. A classification of INFO will also be rewarded when \
correctly applied and is strongly encouraged if information is of poor \
quality, if there is not enough information, or if you are not completely \
confident in your answer.  Provide the output as a plain text separated \
by commas, and provide only the categories listed (either list a \
combination of INDUSTRIAL, ENDOGENOUS, PERSONAL CARE, MEDICAL, FOOD or \
list INFO), with no justification. Provided Information:\n"

# "Note that MEDICAL refers to compounds used as medications in humans or animals"


In [5]:

import asyncio
import requests
import time

def classification_to_bits(classification):
    categories = ["MEDICAL", "FOOD", "INDUSTRIAL", "PERSONAL CARE", "ENDOGENOUS", "INFO"]
    bits = ["1" if category in classification else "0" for category in categories]

    if len(set(classification) - set(categories)) > 0:
        unknown_categories = ",".join(set(classification) - set(categories))
        return "".join(bits)+f"_UNKNOWN({unknown_categories})"
    return "".join(bits)

def bits_to_classification(bits):
    categories = ["MEDICAL", "FOOD", "INDUSTRIAL", "PERSONAL CARE", "ENDOGENOUS", "INFO"]
    classification = [categories[i] for i in range(len(categories)) if bits[i] == "1"]
    return classification


async def async_run(data_list, chemsource_instance):
    # Get the current running event loop instead of get_event_loop()
    loop = asyncio.get_running_loop()
    # Create tasks for each row - now passing tuples of (name, text)
    futures = [loop.run_in_executor(None, chemsource_instance.classify, name, text) for name, text in data_list]
    # Gather all results
    result = await asyncio.gather(*futures, return_exceptions=True)
    return result

async def parallel_classify(data_in, col_header, chem_instance, data_out_path, batch_size=50):

    # Check if output file exists

    if os.path.exists(data_out_path):
        # Check which FEATURE_IDs have already been processed
        processed_ids = set()
        with open(data_out_path, "r") as f:
            next(f)  # Skip header
            for line in f:
                processed_id = line.split(",")[0]
                processed_ids.add(processed_id)
        # Filter data_in to exclude already processed FEATURE_IDs
        data_in = data_in[~data_in["FEATURE_ID"].astype(str).isin(processed_ids)]
        print(f"Resuming from existing file. {len(processed_ids)} entries already processed.")
    else:   
        with open(data_out_path, "w") as f:
            f.write(",".join(data_in.columns.astype(str)))
            f.write(f",{col_header}\n")
    
    feature_ids = data_in["FEATURE_ID"].tolist()
    data_tuples = list(data_in[["NAME", "TEXT"]].itertuples(index=False, name=None))

    for i in range(0, len(data_tuples), batch_size):
        batch_data = data_tuples[i:i+batch_size]
        batch_ids = feature_ids[i:i+batch_size] 
        batch_classifications = await async_run(batch_data, chem_instance)
        # Fixed: use batch_ids[j] instead of feature_ids[i]
        classifications_ids_bits = [str(batch_ids[j]) 
            + "," + classification_to_bits(batch_classifications[j]) + "\n"
            for j in range(len(batch_ids))]
        with open(data_out_path, "a") as f:
            for line in classifications_ids_bits:
                f.write(line)
        time.sleep(1)  # To avoid rate limiting

# non_medical_random_sample_data = list(non_medical_random_sample[["NAME", "TEXT"]].itertuples(index=False, name=None))

# for i in range(num_repeats):

#     classifications = await async_run(non_medical_random_sample_data)
#     time.sleep(1)
#     classifications_bits = [classification_to_bits(c) for c in classifications]
#     with open(data_out_path, "a") as f:
#         f.write(",".join(classifications_bits) + "\n")


In [5]:
chem.prompt = prompt_industrial_strict

await parallel_classify(
    drug_library_text,
    "INDUSTRIAL_STRICT_BITS",
    chem,
    "../data/drug_library/multiple_prompts/industrial_strict_bits.csv",
    batch_size=50
)

Resuming from existing file. 4953 entries already processed.


In [7]:
chem.prompt = prompt_industrial_loose

await parallel_classify(
    drug_library_text,
    "INDUSTRIAL_LOOSE_BITS",
    chem,
    "../data/drug_library/multiple_prompts/industrial_loose_bits.csv",
    batch_size=50
)

Resuming from existing file. 3050 entries already processed.


In [8]:
chem.prompt = prompt_veterinary_looped_in

await parallel_classify(
    drug_library_text,
    "VETERINARY_LOOPED_IN_BITS",
    chem,
    "../data/drug_library/multiple_prompts/veterinary_looped_in_bits.csv",
    batch_size=50
)

Resuming from existing file. 4650 entries already processed.


In [7]:
chem.prompt = prompt_industrial_strict_medical_loose

await parallel_classify(
    drug_library_text,
    "VETERINARY_LOOPED_IN_BITS",
    chem,
    "../data/drug_library/multiple_prompts/industrial_strict_medical_loose_bits.csv",
    batch_size=50
)

In [6]:
chem.prompt = prompt_no_description

await parallel_classify(
    drug_library_text,
    "NO_DESCRIPTION_BITS",
    chem,
    "../data/drug_library/multiple_prompts/no_description_bits.csv",
    batch_size=50
)